# Step 42 QC Alignment Tables S6 S7 And Figure 1 Support

## What This Notebook Does

This is the Stage-4 manuscript-support notebook. It turns the cleaned alignment outputs into scripted support files for:
- Table S6
- Table S7
- Figure 1 support panels

It does not pretend to automate the entire final Figure 1 assembly. The final figure still includes schematic and layout decisions that remain hybrid/manual.

## When To Run It

Run this notebook after:
- `step40_align_eeg_to_bold_trs_and_build_keep_masks.ipynb`
- `step41_build_final_no_lag_fusion_observation_segments.ipynb`

This notebook summarizes the canonical public branch only:
- `FEATURE_MODE = "nolags"`
- `MINLEN = 15`

## Manuscript Linkage

- Main Methods 2.4
- Supplementary Methods 1.5
- Figure 1 support
- Supplementary Tables S6-S7

## Inputs Expected

- Step-40 outputs, especially `qc/alignment_parameters_used.json` and the per-run alignment products
- Step-41 canonical no-lag minlen15 outputs, especially `segments_manifest.tsv` and `qc/per_run_segments_minlen15.csv`

## Outputs Written

- `manuscript_support/table_s6_alignment_parameters.csv`
- `manuscript_support/table_s7_final_dataset_run_summary.csv`
- `manuscript_support/final_dataset_summary.json`
- `manuscript_support/figure1_support/*`

## Manual Or Hybrid Note

This notebook writes the scripted support outputs for Figure 1, but the final manuscript figure still requires hybrid/manual assembly.


In [ ]:
# Step 0. Import Python modules and locate the active Stage-4 helper module

import sys
from pathlib import Path

STAGE4_DIR = Path.cwd()
candidate = Path.cwd() / "notebooks" / "4_alignment"
if not (STAGE4_DIR / "stage4_segment_helpers.py").exists() and candidate.exists():
    STAGE4_DIR = candidate

if not (STAGE4_DIR / "stage4_segment_helpers.py").exists():
    raise FileNotFoundError(
        f"Stage-4 helper not found: {STAGE4_DIR / 'stage4_segment_helpers.py'}"
    )

if str(STAGE4_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(STAGE4_DIR.resolve()))

from stage4_segment_helpers import (
    build_table_s6_alignment_parameters,
    build_table_s7_run_level_summary,
    build_final_dataset_summary,
    build_figure1_support_outputs,
)


In [ ]:
# Step 1. User-editable roots and manuscript-output settings
#
# ALIGNMENT_OUTPUT_DIR should point to the Step-40 output root.
# SEGMENTS_OUTPUT_DIR is the canonical Step-41 no-lag minlen15 dataset root.
# MANUSCRIPT_SUPPORT_DIR is where this notebook writes the table-support CSVs,
# the final dataset summary JSON, and the Figure-1 support outputs.
# REPRESENTATIVE_RUN selects which aligned run to show in the support plot.

ALIGNMENT_OUTPUT_DIR = Path("<SET_ALIGNMENT_OUTPUT_DIR>")
SEGMENTS_OUTPUT_DIR = ALIGNMENT_OUTPUT_DIR / "hmm_segments_minlen15_nolags"
MANUSCRIPT_SUPPORT_DIR = ALIGNMENT_OUTPUT_DIR / "manuscript_support"
FIGURE1_SUPPORT_DIR = MANUSCRIPT_SUPPORT_DIR / "figure1_support"
REPRESENTATIVE_RUN = "sub-16_ses-01"

FEATURE_MODE = "nolags"
MINLEN = 15


In [ ]:
# Step 2. Validate the configured inputs and print a short run summary

def assert_configured_path(path_value, label):
    path_str = str(path_value)
    if "<SET_" in path_str:
        raise ValueError(f"{label} is still using a placeholder path. Edit this notebook before running.")

assert_configured_path(ALIGNMENT_OUTPUT_DIR, "ALIGNMENT_OUTPUT_DIR")

PARAMETERS_JSON = ALIGNMENT_OUTPUT_DIR / "qc" / "alignment_parameters_used.json"
PER_RUN_SEGMENTS_CSV = SEGMENTS_OUTPUT_DIR / "qc" / f"per_run_segments_minlen{MINLEN}.csv"
MANIFEST_TSV = SEGMENTS_OUTPUT_DIR / "segments_manifest.tsv"

print("Stage 4 / Step 42: build manuscript-facing alignment support outputs")
print(f"  Alignment root:       {ALIGNMENT_OUTPUT_DIR}")
print(f"  Canonical segments:   {SEGMENTS_OUTPUT_DIR}")
print(f"  Output support dir:   {MANUSCRIPT_SUPPORT_DIR}")
print(f"  Feature mode:         {FEATURE_MODE}")
print(f"  Minimum segment len:  {MINLEN} TR")
print(f"  Representative run:   {REPRESENTATIVE_RUN}")
print("  Figure note:          final manuscript figure assembly remains hybrid/manual.")


In [ ]:
# Step 3. Build Table S6, Table S7, and the final dataset summary files

MANUSCRIPT_SUPPORT_DIR.mkdir(parents=True, exist_ok=True)

table_s6 = build_table_s6_alignment_parameters(
    alignment_parameters_json=PARAMETERS_JSON,
    feature_mode=FEATURE_MODE,
    minlen=MINLEN,
    out_csv=MANUSCRIPT_SUPPORT_DIR / "table_s6_alignment_parameters.csv",
)

table_s7 = build_table_s7_run_level_summary(
    per_run_segments_csv=PER_RUN_SEGMENTS_CSV,
    out_csv=MANUSCRIPT_SUPPORT_DIR / "table_s7_final_dataset_run_summary.csv",
    minlen=MINLEN,
)

final_dataset_summary = build_final_dataset_summary(
    manifest_tsv=MANIFEST_TSV,
    out_json=MANUSCRIPT_SUPPORT_DIR / "final_dataset_summary.json",
)

table_s7.head(20)


In [ ]:
# Step 4. Build the scripted Figure-1 support outputs

figure1_info = build_figure1_support_outputs(
    alignment_output_dir=ALIGNMENT_OUTPUT_DIR,
    segment_root=SEGMENTS_OUTPUT_DIR,
    representative_run=REPRESENTATIVE_RUN,
    out_dir=FIGURE1_SUPPORT_DIR,
    minlen=MINLEN,
)

print("Saved Table S6:", MANUSCRIPT_SUPPORT_DIR / "table_s6_alignment_parameters.csv")
print("Saved Table S7:", MANUSCRIPT_SUPPORT_DIR / "table_s7_final_dataset_run_summary.csv")
print("Saved final dataset summary:", MANUSCRIPT_SUPPORT_DIR / "final_dataset_summary.json")
print("Saved Figure 1 support manifest:", figure1_info["manifest_json"])
figure1_info
